# VQASynth++ Phase 5 — Stage 1: SFT cold-start (Qwen3-VL-8B-Instruct, q+v LoRA)

Installs metric + grounded priors **and the parseable 4-tag trace format** so the
format-gated GRPO reward (Stage 2) has signal. Base = **Qwen3-VL-8B-Instruct**
(grounding-strong; reasoning installed via these traces, not inherited — the
Thinking variant runs as the contrast arm). Plain `transformers.Trainer` + LoRA
(q_proj, v_proj), matching the `qwen3vl_thinking_qv_balanced_lora` harness.

Consumes the gated VQASynth++ data via `vqasynth.training` + `vqasynth.trace_validation`
(Phases 3–4). Run on a single 8B-capable GPU (A100/H100 or Colab A100).

> Prereq (Stage 0): the staged pipeline has produced rows with `grounded_state` +
> a trace (`output`) + `messages` + `image`. This notebook gates, formats, and SFTs.

## 0. Install (Colab)

In [ ]:
# %pip install -q "transformers>=4.57" accelerate peft datasets pillow qwen-vl-utils
# NOTE: plain transformers.Trainer for SFT — NOT TRL SFTTrainer (chunked-CE VLM+PEFT bug).
# Restart the runtime after install before importing (Colab).

## 1. Config

In [ ]:
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"   # arm #2: "Qwen/Qwen3-VL-8B-Thinking"
DATA_REPO = "remyxai/vqasynthpp_traces"   # Stage-0 pipeline output (grounded_state+trace); set to yours
IMAGE_COL, TRACE_COL, MESSAGES_COL = "image", "output", "messages"
ANSWER_COL, GROUNDED_COL = "answer", "grounded_state"   # answer optional; gate skips checks it can't run

OUTPUT_DIR = "qwen3vl8b_instruct_qv_vqasynthpp_sft"
LORA_TARGETS = ["q_proj", "v_proj"]       # q+v balanced (harness ablation: v-only / late-stack are the arms)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EPOCHS, LR, BSZ, GRAD_ACCUM, MAX_LEN = 1, 1e-4, 1, 8, 2048
SEED = 0

## 2. Load + gate + format the data
Gate with `trace_validation.filter_dataset` (drop hallucinated / unfaithful / unhedged / duplicate rows), then format survivors into chat SFT examples.

In [ ]:
from datasets import load_dataset
from vqasynth import trace_validation as tv
from vqasynth import training as T

raw = load_dataset(DATA_REPO, split="train")
rows = list(raw)

def q_of(r):  return T.extract_question(r.get(MESSAGES_COL))
def tr_of(r): return r.get(TRACE_COL)
def gs_of(r): return r.get(GROUNDED_COL)
def an_of(r): return r.get(ANSWER_COL)

report = tv.filter_dataset(
    rows, output_of=tr_of, answer_of=an_of, grounded_of=gs_of, question_of=q_of,
)
print("GATE:", {k: report[k] for k in ("n_in","n_kept","n_dropped_quality","n_dropped_dup")})
print("drop reasons:", report["drop_reasons"])
print("chance calibration:", report["chance_calibration"])
assert not report["chance_calibration"]["degenerate"], "degenerate answer distribution — fix data before training"

kept = report["kept"]
examples, dropped = T.format_sft_dataset(kept, question_of=q_of, trace_of=tr_of)
# keep the image aligned with each formatted example
images = [kept[i][IMAGE_COL] for i in range(len(kept)) if T.to_sft_example(q_of(kept[i]), tr_of(kept[i]))]
print(f"SFT examples: {len(examples)} (formatter dropped {dropped})")
for ex, img in zip(examples, images):
    ex["image"] = img

## 3. Processor + collator
Apply the chat template, process the image, and **mask the prompt** so loss lands only on the assistant trace. Align specifics with the proven harness collator.

In [ ]:
import torch
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

def collate(batch):
    texts, image_inputs = [], []
    for ex in batch:
        msgs = ex["messages"]
        # inject the real PIL image into the user image placeholder
        for m in msgs:
            if m["role"] == "user":
                for c in m["content"]:
                    if c.get("type") == "image":
                        c["image"] = ex["image"]
        texts.append(processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
        imgs, _ = process_vision_info(msgs)
        image_inputs.append(imgs)
    enc = processor(text=texts, images=image_inputs, return_tensors="pt",
                    padding=True, truncation=True, max_length=MAX_LEN)
    labels = enc["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    # TODO (align w/ harness): also mask the prompt span so loss is assistant-only;
    # the qv_balanced_lora harness does this via the assistant-turn offsets.
    enc["labels"] = labels
    return enc

## 4. Model + LoRA (q+v)

In [ ]:
from transformers import Qwen3VLForConditionalGeneration
from peft import LoraConfig, get_peft_model

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
)
model.config.use_cache = False
peft_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_cfg)
model.print_trainable_parameters()

## 5. Train (plain transformers.Trainer)

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BSZ, gradient_accumulation_steps=GRAD_ACCUM,
    bf16=True, gradient_checkpointing=True, logging_steps=10, save_steps=200,
    lr_scheduler_type="cosine", warmup_ratio=0.03, seed=SEED, report_to=[],
    remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=examples, data_collator=collate)
trainer.train()
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("saved adapter ->", OUTPUT_DIR)

## 6. Sanity check + next
Confirm the model emits the parseable 4-tag format (so Stage-2's format-gated reward has signal), then proceed to **Stage 2 (GRPO)** with `vqasynth.training.make_grpo_reward()` and **Phase 6** (gating eval: 8B base ± VQASynth++, per-dimension, chance-normalized; Instruct vs Thinking arms).

In [ ]:
from vqasynth.spatial_reward import format_reward
ex = examples[0]
msgs = [m for m in ex["messages"] if m["role"] == "user"]
for m in msgs:
    for c in m["content"]:
        if c.get("type") == "image":
            c["image"] = ex["image"]
text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
imgs, _ = process_vision_info(msgs)
inputs = processor(text=[text], images=[imgs], return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=512)
gen = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
print(gen)
print("format parseable (4-tag + scene JSON):", format_reward(gen))